# Human 배치 입력·정답 진단 — 학습 없이 원인 좁히기

## 1. 왜 이 진단을 하는가?

게임 수를 늘리면 Validation은 개선됐지만, 데이터 부족과 현재 표현의 한계가 함께 있을 수 있습니다.
모델을 다시 학습하기 전에 **모델이 실제로 보는 입력이 같을 때 Human 정답도 같은지** 확인합니다.

이번 범위는 기존 Train 149게임의 배치 적격 6,417구간뿐입니다. 신규 수집·데이터 수정·신경망/Scaler fit·Validation/Test payload 읽기를 하지 않습니다.
지난 학습곡선 report는 checksum과 기존 coverage 문맥을 확인하는 용도로만 읽습니다. 전체 실행은 3절에서 직접 켜며 별도 Smoke 학습은 없습니다.

## 2. 무엇을 셀 것인가?

같은 점유 보드와 같은 블록에서 A 배치가 2회, B 배치가 1회라면, 입력만 보는 결정적 모델은 그 3개 중 최대 2개에 일치합니다.
다른 입력의 A 배치 2회를 더하면 관측 5개에서 최대 4개, 즉 80%입니다.

이 값은 **현재 표본에서 동일 입력의 최빈 정답을 고른 경험적 일치 한계**입니다. 미래 정확도의 상한이나 실제 모델 점수가 아닙니다.
사람의 선택 다양성, 모델이 보지 못하는 문맥, 기록 문제를 이 숫자 하나로 구별할 수 없습니다.
동일 입력 반복이 적으면 거의 100%가 나와도 정보가 충분하다는 증거가 아닙니다.

함께 볼 것:

- 배치 종류별 사례 수와 등장 게임 수: 한두 번만 나온 정답이 얼마나 많은가?
- 동일 입력·다중 정답: 블록 색상은 제외하고 기존 모델의 점유 200 + 조각 one-hot 7만 비교합니다.
- 형식 검증: 기존 parser로 20×10 보드·4개 고유 셀·좌표 범위·physical key를 확인합니다.
- 겹침 진단: 정답 셀이 boardBefore 점유와 겹치는가? 관측 시점이 미확인이므로 **오류/부정행위/제외 근거로 자동 판정하지 않습니다**.
- 지난 coverage: 기존 report의 152/1,516 OOV를 문맥으로 표시할 뿐 Validation을 재평가하지 않습니다.

In [ ]:
import copy
import hashlib
import json
import os
import platform
import tempfile
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-input-target-audit"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_input_target_audit.ipynb"
SNAPSHOT_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_snapshot_baseline.ipynb"
SNAPSHOT_DEFS_SHA = "233ceca8322bd4335b798cdd452064d389e0ddbce6451610c7ba7d56a1dd9c5c"
REFERENCE_SHA = "974f23947e1f4b0f87fb581f658a365e13a7de88f4d0384d75a52d193dfb28e4"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(SNAPSHOT_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == SNAPSHOT_DEFS_SHA
shared = {"__name__": "input_target_audit_shared"}
for cell in selected:
    exec(compile(cell["source"], str(SNAPSHOT_NOTEBOOK), "exec"), shared)
shared["EXPERIMENT_ID"], shared["SOURCE_NOTEBOOK"] = EXPERIMENT_ID, SOURCE_NOTEBOOK
shared["base"]["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "repo_input",
             "placement_row", "physical_key", "report_path", "publish_report", "reject_constant", "finite_float"):
    globals()[name] = shared[name]
MANIFEST_SHA = shared["MANIFEST_SHA"]
TRAIN_SHA = shared["EXPECTED"]["train"][-1]


In [ ]:
def load_train():
    """Read only the pinned manifest, Train payload and previous report."""
    directory = repo_input(os.environ.get("HUMAN_PLACEMENT_DATASET",
        "data/datasets/human-placement-action-offline-v1-20261102-run1"))
    paths = {"manifest": directory / "manifest.json", "train": directory / "train.jsonl",
             "reference": repo_input(os.environ.get("HUMAN_PLACEMENT_INPUT_AUDIT_REFERENCE",
                "data/ml-experiments/human-placement-game-count-learning-curve/full-report-run1.json"))}
    watched = {p: sha256_file(p) for p in paths.values()}
    require([watched[paths[k]] for k in ("manifest", "train", "reference")]
            == [MANIFEST_SHA, TRAIN_SHA, REFERENCE_SHA], "INPUT_CHECKSUM_MISMATCH")
    manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"), parse_constant=reject_constant)
    reference = json.loads(paths["reference"].read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(manifest["status"] == "offline_dataset_constructed" and manifest["policy"] == "offline-use-v1"
            and manifest["testPayloadRead"] is False and manifest["files"]["train.jsonl"]["sha256"] == TRAIN_SHA,
            "DATASET_CONTRACT_INVALID")
    require(reference["experimentId"] == "human-placement-game-count-learning-curve"
            and reference["runMode"] == "full" and reference["fullCohortCovered"] is True
            and reference["baselineReproduced"] is True and reference["fitWarnings"] == 0
            and reference["provenance"]["manifestSha256"] == MANIFEST_SHA
            and reference["provenance"]["datasetFiles"]["train"] == TRAIN_SHA, "REFERENCE_CONTRACT_INVALID")
    rows, raw_count, ineligible = [], 0, 0
    with paths["train"].open("rb") as handle:
        for line in handle:
            segment = json.loads(line, parse_constant=reject_constant, parse_float=finite_float)
            raw_count += 1
            eligible = segment["eligibility"]["placement_snapshot_eligible"]["eligible"]
            require(type(eligible) is bool and segment["lineage"]["split"] == "train", "TRAIN_ROW_CONTRACT_INVALID")
            if eligible:
                rows.append(placement_row(segment, "train"))
            else:
                ineligible += 1
    rows.sort(key=lambda r: (r["sourceLine"], r["segmentId"]))
    games = {r["gameId"] for r in rows}
    reserved = set(manifest["provenance"]["reservedTestGameIds"])
    validation_games = {v["gameId"] for v in reference["results"]["fraction_100"]["validationAll"]["perGame"]}
    require((raw_count, len(rows), len(games), ineligible) == (6440, 6417, 149, 23)
            and len({r["segmentId"] for r in rows}) == len(rows), "TRAIN_COHORT_CHANGED")
    require(games == set(reference["results"]["fraction_100"]["gameIds"])
            and len(validation_games) == 31 and len(reserved) == 33
            and not games & (validation_games | reserved) and not validation_games & reserved, "GAME_SPLIT_LEAKAGE")
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    context = {"source": "previous_learning_curve_report", "recomputed": False,
               "validationSegments": 1516,
               "coverage": reference["results"]["fraction_100"]["validationAll"]["segmentMicro"]["knownTargetCoverage"],
               "unknownTargetSegments": reference["results"]["fraction_100"]["validationAll"]["segmentMicro"]["unknownTargetSegments"]}
    require(context["unknownTargetSegments"] == 152
            and reference["results"]["fraction_100"]["validationAll"]["segmentMicro"]["segments"] == 1516,
            "REFERENCE_COVERAGE_CHANGED")
    return rows, watched, context


In [ ]:
def input_key(row):
    """Use exactly the raw 207 binary features seen by the placement baseline."""
    x = np.asarray(row["x"])
    require(x.shape == (207,) and np.isfinite(x).all() and np.isin(x, [0, 1]).all()
            and x[200:].sum() == 1 and row["pieceType"] in shared["PIECES"]
            and x[200 + shared["PIECES"].index(row["pieceType"])] == 1, "INPUT_FEATURE_INVALID")
    return tuple(int(v) for v in x)

def audit_rows(rows):
    """Count observed ambiguity and support; do not fit, simulate, reject or relabel."""
    require(rows and all(r["split"] == "train" for r in rows)
            and len({r["segmentId"] for r in rows}) == len(rows), "AUDIT_TRAIN_ONLY_OR_DUPLICATE")
    groups, classes = defaultdict(list), defaultdict(list)
    overlap_segments = overlap_cells = 0
    for row in rows:
        key = input_key(row)
        physical = physical_key({"pieceType": row["pieceType"], "rotation": 0, "row": 0, "col": 0,
                                 "cells": row["targetCells"]}, selected=True)
        require(physical == row["targetKey"], "TARGET_CLASS_MISMATCH")
        groups[key].append(row)
        classes[row["targetKey"]].append(row)
        overlap = sum(key[c["row"] * 10 + c["col"]] for c in row["targetCells"])
        overlap_segments += int(overlap > 0)
        overlap_cells += overlap
    details, repeated, ambiguous_games = [], [], set()
    majority_total = 0
    for key, members in groups.items():
        counts = Counter(r["targetKey"] for r in members)
        majority_total += max(counts.values())
        digest = hashlib.sha256(bytes(key)).hexdigest()
        alternatives = []
        for target, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
            matches = [r for r in members if r["targetKey"] == target]
            representative = min(matches, key=lambda r: (r["sourceLine"], r["segmentId"]))
            alternatives.append({"targetKey": target, "segments": count,
                "games": len({r["gameId"] for r in matches}), "targetCells": representative["targetCells"],
                "exampleSegmentId": representative["segmentId"], "exampleSourceLine": representative["sourceLine"]})
        detail = {"inputSha256": digest, "pieceType": members[0]["pieceType"], "segments": len(members),
                  "games": len({r["gameId"] for r in members}), "targetClasses": len(counts),
                  "majorityCount": max(counts.values()), "alternatives": alternatives,
                  "boardOccupancy": np.array(key[:200]).reshape(20, 10).tolist()}
        if len(members) > 1:
            repeated.append(detail)
        if len(counts) > 1:
            details.append(detail)
            ambiguous_games.update(r["gameId"] for r in members)
    details.sort(key=lambda d: (-d["segments"], d["inputSha256"]))
    support = [{"targetKey": key, "segments": len(members),
                "games": len({r["gameId"] for r in members}),
                "uniqueInputs": len({input_key(r) for r in members})}
               for key, members in sorted(classes.items())]
    ranges = [("1", 1, 1), ("2", 2, 2), ("3-5", 3, 5), ("6-10", 6, 10), ("11+", 11, len(rows))]
    buckets = [{"support": name, "classes": sum(lo <= s["segments"] <= hi for s in support),
                "segments": sum(s["segments"] for s in support if lo <= s["segments"] <= hi)}
               for name, lo, hi in ranges]
    repeated_n = sum(d["segments"] for d in repeated)
    counts = {"trainSegments": len(rows), "trainGames": len({r["gameId"] for r in rows}),
              "targetClasses": len(support), "inputGroups": len(groups),
              "repeatedInputGroups": len(repeated), "repeatedInputSegments": repeated_n,
              "ambiguousInputGroups": len(details), "ambiguousSegments": sum(d["segments"] for d in details),
              "ambiguousGames": len(ambiguous_games),
              "crossGameAmbiguousInputGroups": sum(d["games"] > 1 for d in details),
              "minimumObservedDeterministicDisagreements": len(rows) - majority_total,
              "observedDeterministicAgreement": majority_total / len(rows),
              "repeatedInputObservedAgreement": sum(d["majorityCount"] for d in repeated) / repeated_n if repeated_n else None,
              "classesWithOneOrTwoSegments": sum(s["segments"] <= 2 for s in support),
              "segmentsInOneOrTwoSupportClasses": sum(s["segments"] for s in support if s["segments"] <= 2),
              "classesSeenInOneGame": sum(s["games"] == 1 for s in support),
              "storedShapeValidatedSegments": len(rows), "overlapSegments": overlap_segments,
              "overlapCells": overlap_cells}
    require(sum(b["classes"] for b in buckets) == len(support)
            and sum(b["segments"] for b in buckets) == len(rows), "SUPPORT_DENOMINATOR_CHANGED")
    fallback = sorted(repeated, key=lambda d: (-d["segments"], d["inputSha256"]))
    if not fallback:
        first = min(rows, key=lambda r: (r["sourceLine"], r["segmentId"]))
        fallback = [{"inputSha256": hashlib.sha256(bytes(input_key(first))).hexdigest(),
                     "pieceType": first["pieceType"], "segments": 1, "games": 1, "targetClasses": 1,
                     "majorityCount": 1, "alternatives": [{"targetKey": first["targetKey"], "segments": 1,
                        "games": 1, "targetCells": first["targetCells"], "exampleSegmentId": first["segmentId"],
                        "exampleSourceLine": first["sourceLine"]}],
                     "boardOccupancy": np.array(first["x"][:200], dtype=int).reshape(20, 10).tolist()}]
    return {"counts": counts, "supportBuckets": buckets, "classSupport": support,
            "inputMultiplicityHistogram": [{"segmentsPerInput": n, "inputGroups": count}
                                           for n, count in sorted(Counter(map(len, groups.values())).items())],
            "ambiguousGroups": details, "examples": (details or fallback)[:3]}


In [ ]:
def run_full():
    before_code = definition_sha(SOURCE_NOTEBOOK)
    rows, watched, context = load_train()
    diagnostics = audit_rows(rows)
    require(diagnostics["counts"]["targetClasses"] == 2133, "TRAIN_VOCABULARY_CHANGED")
    vocabulary = [s["targetKey"] for s in diagnostics["classSupport"]]
    require(hashlib.sha256(json_bytes(vocabulary)).hexdigest()
            == "26edffc2ac5501450f78e976818cba195882d558c41530ff2196a0b017d45084", "TRAIN_VOCABULARY_SHA_CHANGED")
    require(all(sha256_file(p) == h for p, h in watched.items())
            and definition_sha(SOURCE_NOTEBOOK) == before_code
            and definition_sha(SNAPSHOT_NOTEBOOK) == SNAPSHOT_DEFS_SHA, "INPUT_OR_CODE_CHANGED")
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": "full",
            "executionPolicy": "full-only", "analysisStatus": "input_target_audit_requires_review",
            "fullCohortCovered": True, "policyDecisionEligible": True, "canonicalResultEligible": False,
            "productionIntegration": False, "inferenceEligible": False, "wholePolicyEvidence": False,
            "modelFitCount": 0, "scalerFitCount": 0, "validationPayloadRead": False,
            "testFileRead": False, "rawFileRead": False, "dataModified": False, "dataExcludedByAudit": 0,
            "snapshotObservation": "unverified", "legalityChecked": False, "reachableCandidatesGenerated": False,
            "diagnostics": diagnostics, "previousCoverageContext": context,
            "interpretationContract": {"agreement": "empirical_same_input_majority_not_population_ceiling",
                "overlap": "conditional_on_prelock_observation_not_an_automatic_error",
                "labelAmbiguity": "not_automatically_bad_data", "zeroAmbiguity": "not_proof_of_sufficient_information",
                "causalAttribution": "not_established", "neuralPerformanceMeasured": False},
            "sourceDefinitionSha256": before_code, "snapshotDefinitionSha256": SNAPSHOT_DEFS_SHA,
            "referenceSha256": REFERENCE_SHA, "sourceUnchanged": True,
            "provenance": {"manifestSha256": MANIFEST_SHA, "trainSha256": TRAIN_SHA,
                           "trainRows": 6440, "placementEligibleRows": 6417, "previouslyIneligibleRows": 23,
                           "trainValidationGameIdOverlap": 0, "trainReservedTestGameIdOverlap": 0,
                           "splitEvidence": "manifest_test_ids_and_previous_report_validation_ids"},
            "environment": {"python": platform.python_version(), "numpy": np.__version__,
                            "pandas": pd.__version__}}

def check_report(report):
    require(report["reportSchemaVersion"] == 1 and report["experimentId"] == EXPERIMENT_ID
            and report["runMode"] == "full" and report["fullCohortCovered"] is True
            and report["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and report["snapshotDefinitionSha256"] == SNAPSHOT_DEFS_SHA
            and report["referenceSha256"] == REFERENCE_SHA
            and report["modelFitCount"] == 0 and report["scalerFitCount"] == 0
            and report["validationPayloadRead"] is False and report["testFileRead"] is False
            and report["rawFileRead"] is False and report["dataModified"] is False
            and report["dataExcludedByAudit"] == 0 and report["canonicalResultEligible"] is False
            and report["productionIntegration"] is False and report["inferenceEligible"] is False
            and report["provenance"]["manifestSha256"] == MANIFEST_SHA
            and report["provenance"]["trainSha256"] == TRAIN_SHA, "STALE_OR_DIFFERENT_REPORT")
    c = report["diagnostics"]["counts"]
    require((c["trainGames"], c["trainSegments"], c["targetClasses"]) == (149, 6417, 2133)
            and 0 <= c["minimumObservedDeterministicDisagreements"] <= c["ambiguousSegments"] <= c["trainSegments"]
            and np.isclose(c["observedDeterministicAgreement"],
                           1 - c["minimumObservedDeterministicDisagreements"] / c["trainSegments"]),
            "REPORT_COUNTS_INVALID")


In [ ]:
def synthetic_rows():
    def segment(index, target_col=0, occupied=False, color="bar"):
        board = [[None] * 10 for _ in range(20)]
        if occupied:
            board[18][0] = color
        cells = [{"row": 19, "col": target_col + i} for i in range(4)]
        key = physical_key({"pieceType": "bar", "rotation": 0, "row": 0, "col": 0, "cells": cells}, selected=True)
        return {"segmentId": f"s{index}", "lineage": {"split": "train", "gameId": f"G{index % 2}", "sourceLine": index + 1},
                "support": {"policy": "offline-use-v1"},
                "placement": {"offlineOnly": True, "inferenceEligible": False,
                    "inputSnapshot": {"boardBefore": board, "activePiece": {"pieceType": "bar"}},
                    "target": {"selectedCells": cells, "physicalClassKey": key}}}
    return [placement_row(s, "train") for s in
            [segment(0), segment(1), segment(2, 1), segment(3, occupied=True),
             segment(4, occupied=True, color="tree")]]

def internal_checks():
    rows = synthetic_rows()
    before = copy.deepcopy(rows)
    result = audit_rows(rows)
    c = result["counts"]
    assert (c["trainSegments"], c["inputGroups"], c["repeatedInputGroups"],
            c["ambiguousInputGroups"], c["ambiguousSegments"]) == (5, 2, 2, 1, 3)
    assert c["minimumObservedDeterministicDisagreements"] == 1 and c["observedDeterministicAgreement"] == .8
    assert c["repeatedInputObservedAgreement"] == .8 and c["crossGameAmbiguousInputGroups"] == 1
    assert result == audit_rows(list(reversed(rows)))
    assert all(np.array_equal(a["x"], b["x"]) and a["targetCells"] == b["targetCells"]
               for a, b in zip(rows, before))
    assert input_key(rows[3]) == input_key(rows[4])
    assert [b["classes"] for b in result["supportBuckets"]] == [1, 0, 1, 0, 0]
    single = audit_rows(rows[:1])
    assert single["counts"]["repeatedInputObservedAgreement"] is None
    assert single["counts"]["ambiguousInputGroups"] == 0 and single["counts"]["observedDeterministicAgreement"] == 1
    overlap = copy.deepcopy(rows[:1])
    overlap[0]["x"][190] = 1
    assert audit_rows(overlap)["counts"]["overlapSegments"] == 1
    bad_rows = []
    bad = copy.deepcopy(rows[:1]); bad[0]["split"] = "validation"; bad_rows.append(bad)
    bad = copy.deepcopy(rows[:1]); bad[0]["targetKey"] = "bar|wrong"; bad_rows.append(bad)
    bad = copy.deepcopy(rows[:1]); bad[0]["targetCells"][1] = bad[0]["targetCells"][0]; bad_rows.append(bad)
    bad = copy.deepcopy(rows[:1]); bad[0]["x"][0] = np.nan; bad_rows.append(bad)
    bad = copy.deepcopy(rows[:1]); bad[0]["targetCells"][0]["row"] = 20; bad_rows.append(bad)
    bad_rows.append(rows[:1] * 2)
    for bad in bad_rows:
        try:
            audit_rows(bad)
        except ValueError:
            pass
        else:
            raise AssertionError("INVALID_AUDIT_ROW_ACCEPTED")
    with tempfile.TemporaryDirectory(dir="/private/tmp") as directory:
        root = Path(directory)
        existing = root / "existing.json"; existing.write_text("{}", encoding="utf-8")
        alias = root / "alias.json"; alias.symlink_to(existing)
        dangling = root / "dangling.json"; dangling.symlink_to(root / "missing.json")
        hard = root / "hard.json"; os.link(existing, hard)
        for path in (existing, alias, dangling, hard, root / ".." / "bad.json", SOURCE_NOTEBOOK):
            try:
                report_path(str(path))
            except ValueError:
                pass
            else:
                raise AssertionError("UNSAFE_OUTPUT_ACCEPTED")
    return "PASS: empirical ambiguity math; exact input/color equivalence; support counts; order/input immutability; shape/NaN/split failures; output protection. No dataset access or fit."


In [ ]:
def show_diagnostics(diagnostics, context=None):
    c = diagnostics["counts"]
    display(pd.DataFrame([c]).T.rename(columns={0: "value"}))
    support = pd.DataFrame(diagnostics["supportBuckets"]).set_index("support")
    support["classFraction"] = support["classes"] / c["targetClasses"]
    support["segmentFraction"] = support["segments"] / c["trainSegments"]
    display(support)
    display(pd.DataFrame(diagnostics["classSupport"]).sort_values(["segments", "targetKey"]).head(12))
    display(pd.DataFrame(diagnostics["inputMultiplicityHistogram"]))
    if context:
        print("Previous report context only; Validation was NOT read or re-evaluated.")
        display(pd.DataFrame([context]))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    support["classes"].plot.bar(ax=axes[0], rot=0)
    axes[0].set(title="Train target-class support", xlabel="Segments per target class", ylabel="Target classes")
    pd.Series({"Unique-input segments": c["trainSegments"] - c["repeatedInputSegments"],
               "Repeated, one target": c["repeatedInputSegments"] - c["ambiguousSegments"],
               "Repeated, multiple targets": c["ambiguousSegments"]}).plot.bar(ax=axes[1], rot=20)
    axes[1].set(title="Exact 207-feature input groups", ylabel="Train segments")
    pd.Series({"Same-input majority agreements": c["trainSegments"] - c["minimumObservedDeterministicDisagreements"],
               "Observed unavoidable disagreements": c["minimumObservedDeterministicDisagreements"]}).plot.bar(ax=axes[2], rot=15)
    axes[2].set(title="Empirical deterministic fit limit", ylabel="Train segments")
    fig.suptitle("Dataset diagnostics only: NOT neural accuracy, NOT a future-performance ceiling")
    fig.tight_layout(); plt.show(); plt.close(fig)
    examples = diagnostics["examples"]
    fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 6), squeeze=False)
    colors = plt.get_cmap("tab10")
    for ax, example in zip(axes.flat, examples):
        ax.imshow(example["boardOccupancy"], cmap="Greys", vmin=0, vmax=1, origin="upper")
        for i, alternative in enumerate(example["alternatives"][:4]):
            cells = alternative["targetCells"]
            ax.scatter([v["col"] for v in cells], [v["row"] for v in cells], marker="s", s=110,
                       facecolors="none", edgecolors=[colors(i)], linewidths=2,
                       label=f"Target {i + 1}: n={alternative['segments']}")
        ax.set(title=f"{example['pieceType']}: {example['segments']} segments / {example['targetClasses']} targets",
               xlim=(-.5, 9.5), ylim=(19.5, -.5), xlabel="Column", ylabel="Row")
        ax.legend(fontsize=8)
    fig.suptitle("Stored board occupancy + Human target cells; observation/reachability UNVERIFIED")
    fig.tight_layout(); plt.show(); plt.close(fig)


## 3. Full: Train 전체 진단

`RUN_FULL=True`로 바꾼 뒤 설정 → 실행 → 4절 결과 셀을 실행합니다. 학습은 전혀 하지 않습니다.
이 실험의 Full은 149게임 전체를 센다는 뜻입니다. Validation/Test payload·raw·운영 DB는 열지 않습니다.

기존 report를 덮어쓰지 않습니다. 재실행하려면 `full-report-run2.json`처럼 새 이름을 사용하세요.
기본값 False에서는 짧은 가상 assert만 실행하고 실제 dataset/reference는 읽지 않습니다.

In [ ]:
RUN_FULL = False
FULL_REPORT = os.environ.get("HUMAN_PLACEMENT_INPUT_AUDIT_REPORT",
    "data/ml-experiments/human-placement-input-target-audit/full-report-run1.json")


In [ ]:
print(internal_checks())
full_report = None
if RUN_FULL:
    destination = report_path(FULL_REPORT)
    full_report = run_full()
    check_report(full_report)
    publish_report(full_report, destination)
    print("Saved:", str(destination), "| SHA-256:", sha256_file(destination))
else:
    print("Full disabled. Synthetic checks only; no real dataset/reference access or model/scaler fit.")


## 4. 결과를 어떻게 읽는가?

- **Support 표:** 사례 1~2개인 class가 많으면 희소성 근거입니다. 이것만으로 성능 저하의 인과를 확정하지 않습니다.
- **동일 입력·다중 정답:** 모든 segment 수와 반복 입력만의 분모를 함께 봅니다. 거의 없는 경우에도 미관측 문맥이 충분하다는 뜻은 아닙니다.
- **경험적 일치 한계:** 같은 입력의 최빈 정답을 아는 가상의 결정적 정책입니다. 실제 모델 정확도·미래 상한으로 쓰지 않습니다.
- **보드 그림:** 회색은 저장된 점유, 테두리는 같은 입력에서 사람이 고른 서로 다른 정답입니다. 최대 4개 정답만 그리며 report에는 전부 남습니다.
- **Overlap:** pre-lock 관측이 맞다는 가정 아래의 겹침 신호입니다. 자동 정제/제외·게임 규칙 검증으로 해석하지 않습니다.
- **과거 coverage:** 이전 report 값의 재표시입니다. 새 Validation 측정이나 Test 증거가 아닙니다.

결과만 다시 보려면 Full을 끄고 아래 `REPORT_TO_READ`에 생성된 JSON 경로를 넣습니다.

In [ ]:
REPORT_TO_READ = None
if REPORT_TO_READ:
    path = repo_input(REPORT_TO_READ)
    loaded = json.loads(path.read_text(encoding="utf-8"), parse_constant=reject_constant)
    check_report(loaded)
    full_report = loaded


In [ ]:
if full_report is not None:
    check_report(full_report)
    print(full_report["analysisStatus"], "| modelFitCount=0 | no automatic exclusions")
    show_diagnostics(full_report["diagnostics"], full_report["previousCoverageContext"])
else:
    print("No Full report loaded. Enable section 3 or set REPORT_TO_READ in section 4.")


## 5. 한계와 다음 gate

이 진단은 기존 모델 입력의 희소성·관측 표본의 다중 정답·저장 형식만 검사합니다.
관측 시점·과거 build/ruleset·물리 geometry·합법성·도달 가능성·현재 40행 runtime 호환성은 여전히 미검증입니다.
게임별 split은 manifest의 Test ID와 지난 report의 Validation ID 경계만 대조하며 Validation/Test payload는 읽지 않습니다.
기존 timing Test가 이미 관측된 사실도 바뀌지 않습니다.

결과를 보고 입력/정답의 한계를 해석한 뒤에만 반복 학습곡선 또는 구조 비교 중 다음 질문을 선택합니다.
새 로그 수집·추가 학습·정답 변경·dataset 제외/재생성·runtime·commit을 자동 시작하지 않습니다.
관측 한계와 실제 인간의 선택 다양성을 구분할 수 없으면 미확인으로 남깁니다.